# SCRC with Dynamic Capability Alpha Allocation

**Research Question**: Does *Inverse Excess AUC* alpha allocation prevent the
catastrophic FPR collapse seen in uniform allocation under compound shift?

## Pipeline

**Stage 1 — Global Deferral** (β-fraction of most-uncertain samples sent to expert)
**Stage 2 — Per-pathology Weighted CRC** (calibrates λ_k* so weighted FNR ≤ α_k)

## Alpha Strategies Compared

| Strategy | Formula | Intuition |
|---|---|---|
| **Uniform** | α_k = α | All pathologies get equal budget |
| **Linear** | α_k ∝ (1 − AUC_k) | Harder pathologies get more slack |
| **Inverse Excess** *(proposed)* | α_k ∝ 1 / max(AUC_k − 0.5, 0.001) | Near-random → very loose; confident → tight |
| **Softmax** | α_k ∝ exp(−AUC_k / T) | Sharp concentration on weakest pathologies |

**Budget Neutrality**: All strategies satisfy mean(α_k) = α_target = 0.10.


In [ ]:
import sys
from pathlib import Path

ROOT = Path('/Users/amo/programData/wcp-l2d')
sys.path.insert(0, str(ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

from wcp_l2d.features import ExtractedFeatures
from wcp_l2d.pathologies import COMMON_PATHOLOGIES
from wcp_l2d.dre import AdaptiveDRE
from wcp_l2d.gnn import build_adjacency_matrix, train_gnn
from wcp_l2d.scrc import (
    multilabel_entropy,
    select_for_deferral,
    calibrate_per_pathology_crc_fnr,
)

print(f"PyTorch: {torch.__version__}")
print(f"Device : {'mps' if torch.backends.mps.is_available() else 'cpu'}")


In [ ]:
SEED         = 42
BETA         = 0.15    # Stage 1 global deferral budget
ALPHA_TARGET = 0.10    # FNR budget to distribute across pathologies
K            = len(COMMON_PATHOLOGIES)
PATHOLOGIES  = COMMON_PATHOLOGIES
DEVICE       = 'mps' if torch.backends.mps.is_available() else 'cpu'

FEAT_DIR   = ROOT / 'data' / 'features'
REPORT_DIR = ROOT / 'report'
REPORT_DIR.mkdir(exist_ok=True)

rng = np.random.RandomState(SEED)

print(f"Pathologies ({K}): {PATHOLOGIES}")
print(f"β = {BETA},  α_target = {ALPHA_TARGET}")


## 1. Data Loading

**CheXpert** (source): 60% train / 20% calibration / 20% ignored
**NIH** (target): 50% unlabelled DRE pool / 50% labelled test
Same random seed as `scrc_hard_fnr.ipynb` for reproducibility.


In [ ]:
chex = ExtractedFeatures.load(FEAT_DIR / 'chexpert_densenet121-res224-chex_features.npz')
nih  = ExtractedFeatures.load(FEAT_DIR / 'nih_densenet121-res224-chex_features.npz')

# CheXpert split
N_chex = len(chex.features)
idx    = rng.permutation(N_chex)
n_tr   = int(0.60 * N_chex)
n_cal  = int(0.20 * N_chex)

X_train_raw = chex.features[idx[:n_tr]]
Y_train     = chex.labels  [idx[:n_tr]]
X_cal_raw   = chex.features[idx[n_tr : n_tr + n_cal]]
Y_cal       = chex.labels  [idx[n_tr : n_tr + n_cal]]

# NIH split
N_nih    = len(nih.features)
nih_perm = rng.permutation(N_nih)
n_pool   = N_nih // 2

X_pool_raw = nih.features[nih_perm[:n_pool]]
X_nih_raw  = nih.features[nih_perm[n_pool:]]
Y_nih_test = nih.labels  [nih_perm[n_pool:]]

# Scale with CheXpert train statistics
scaler  = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_cal   = scaler.transform(X_cal_raw)
X_pool  = scaler.transform(X_pool_raw)
X_nih   = scaler.transform(X_nih_raw)

print(f"CheXpert  train: {X_train.shape}  cal: {X_cal.shape}")
print(f"NIH       pool:  {X_pool.shape}   test: {X_nih.shape}")


## 2. GNN Training

Train a **LabelGCN** (ML-GCN variant) on CheXpert features:
- 2-layer GCN over label co-occurrence graph
- Residual blending with binary LR baseline (α = 0.7)
- 50 epochs, best-epoch restore, MPS backend


In [ ]:
# Binary logistic regressors for residual initialisation
lr_classifiers = []
for k in range(K):
    valid_k = ~np.isnan(Y_train[:, k])
    lr_k = LogisticRegression(solver='lbfgs', max_iter=500)
    lr_k.fit(X_train[valid_k], Y_train[valid_k, k].astype(int))
    lr_classifiers.append(lr_k)

# decision_function gives log-odds (correct init logits for LabelGCN)
init_logits_train = np.column_stack([clf.decision_function(X_train) for clf in lr_classifiers])
init_logits_val   = np.column_stack([clf.decision_function(X_cal)   for clf in lr_classifiers])
init_logits_pool  = np.column_stack([clf.decision_function(X_pool)  for clf in lr_classifiers])
init_logits_nih   = np.column_stack([clf.decision_function(X_nih)   for clf in lr_classifiers])

print(f"LR classifiers fitted: {len(lr_classifiers)}")
print(f"Init logits (train): {init_logits_train.shape}")


In [ ]:
A = build_adjacency_matrix(Y_train, tau=0.10)

print("Training LabelGCN (50 epochs, verbose=False)...")
gnn, history = train_gnn(
    features_train    = X_train,
    labels_train      = Y_train,
    features_val      = X_cal,
    labels_val        = Y_cal,
    adjacency         = A,
    init_logits_train = init_logits_train,
    init_logits_val   = init_logits_val,
    feat_dim          = 1024,
    embed_dim         = 300,
    hidden_dim        = 1024,
    alpha             = 0.7,
    epochs            = 50,
    save_best         = True,
    batch_size        = 512,
    lr                = 1e-3,
    weight_decay      = 1e-4,
    device            = DEVICE,
    verbose           = False,
)
best_ep  = history['best_epoch'][0]
best_auc = max(history['val_auc'])
print(f"Done. Best epoch: {best_ep}/50  val AUC: {best_auc:.4f}")


In [ ]:
def get_probs(model, X_np, init_np, batch_size=2048):
    """Batch inference → sigmoid probabilities."""
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb = torch.tensor(X_np[i:i+batch_size], dtype=torch.float32)
            lb = torch.tensor(init_np[i:i+batch_size], dtype=torch.float32)
            out.append(torch.sigmoid(model(xb, lb)).numpy())
    return np.vstack(out)

gnn.cpu()
p_cal  = get_probs(gnn, X_cal,  init_logits_val)
p_pool = get_probs(gnn, X_pool, init_logits_pool)
p_nih  = get_probs(gnn, X_nih,  init_logits_nih)

# NIH test AUC per pathology
gnn_aucs = np.zeros(K)
for k in range(K):
    valid_k = ~np.isnan(Y_nih_test[:, k])
    yv, pv  = Y_nih_test[valid_k, k], p_nih[valid_k, k]
    gnn_aucs[k] = roc_auc_score(yv, pv) if len(np.unique(yv)) > 1 else np.nan

print(f"{'Pathology':<22} {'NIH AUC':>8}")
print("-" * 32)
for name, auc in zip(PATHOLOGIES, gnn_aucs):
    print(f"  {name:<20} {auc:>8.4f}")
print(f"\n  Mean AUC: {np.nanmean(gnn_aucs):.4f}")


## 3. Density Ratio Estimation

**GNN-DRE**: fit importance weights in the 7-dim GNN probability space (no PCA).
- Clipping at 20.0 → ESS ≈ 32% (much better than raw 1024-dim LR-DRE at 6%)
- These weights correct for covariate shift CheXpert → NIH in Stage 2 calibration.


In [ ]:
dre = AdaptiveDRE(n_components=None, weight_clip=20.0, random_state=SEED)
dre.fit(source_features=p_cal, target_features=p_pool)

w_cal = dre.compute_weights(p_cal)   # [N_cal] — used in Stage 2 calibration

diag = dre.diagnostics(p_cal)
print(f"GNN-DRE (clip=20.0):")
print(f"  Domain AUC  : {diag.domain_auc:.4f}")
print(f"  ESS         : {diag.ess:.1f}  ({diag.ess_fraction*100:.1f}%)")
print(f"  Weight max  : {diag.weight_max:.2f}")
print(f"  Weight mean : {diag.weight_mean:.4f} \u00b1 {diag.weight_std:.4f}")


## 4. Alpha Allocation Strategies (Task 1)

Four strategies for distributing the global FNR budget α = 0.10 across K = 7 pathologies.
All satisfy **budget neutrality**: mean(α_k) = α_target.


In [ ]:
def normalize_alphas(alphas: np.ndarray, target: float) -> np.ndarray:
    """Scale allocation so mean = target, clip to [0.01, 0.99]."""
    alphas = np.asarray(alphas, dtype=float).copy()
    m = alphas.mean()
    if m > 0:
        alphas *= target / m
    return np.clip(alphas, 0.01, 0.99)


def alloc_uniform(K: int, alpha_target: float) -> np.ndarray:
    """Uniform: equal budget for all pathologies (baseline)."""
    return np.full(K, alpha_target)


def alloc_linear(aucs: np.ndarray, alpha_target: float) -> np.ndarray:
    """Linear: alpha proportional to (1 - AUC)."""
    raw = 1.0 - np.asarray(aucs)
    return normalize_alphas(raw, alpha_target)


def alloc_inverse_excess(aucs: np.ndarray, alpha_target: float) -> np.ndarray:
    """Inverse Excess AUC (proposed): alpha inversely proportional to AUC advantage over chance.

    excess_k = max(AUC_k - 0.5, 0.001)
    alpha_k  proportional to 1 / excess_k

    High-AUC pathologies (confident) get tight budgets.
    Near-random pathologies get loose budgets -> low lambda -> catch more true positives.
    """
    excess = np.maximum(np.asarray(aucs) - 0.5, 0.001)
    raw    = 1.0 / excess
    return normalize_alphas(raw, alpha_target)


def alloc_softmax(aucs: np.ndarray, alpha_target: float, T: float = 0.05) -> np.ndarray:
    """Softmax: alpha proportional to exp(-AUC / T). Sharp concentration on low-AUC."""
    raw = np.exp(-np.asarray(aucs) / T)
    return normalize_alphas(raw, alpha_target)


print("Allocation functions defined.")


In [ ]:
allocations = {
    'Uniform'        : alloc_uniform(K, ALPHA_TARGET),
    'Linear'         : alloc_linear(gnn_aucs, ALPHA_TARGET),
    'Inverse_Excess' : alloc_inverse_excess(gnn_aucs, ALPHA_TARGET),
    'Softmax'        : alloc_softmax(gnn_aucs, ALPHA_TARGET),
}

# Display table
rows = {'NIH AUC': gnn_aucs}
rows.update(allocations)
alloc_df = pd.DataFrame(rows, index=PATHOLOGIES)
alloc_df.loc['MEAN'] = {k: (v.mean() if isinstance(v, np.ndarray) else np.nan)
                        for k, v in rows.items()}

print("Alpha allocations per pathology (budget-neutral: mean = 0.10):")
print(alloc_df.round(4).to_string())
print()
print("Per-strategy mean alpha (should all ≈ 0.10):")
for name, arr in allocations.items():
    print(f"  {name:<20}: {arr.mean():.4f}")


## 5. SCRC Pipeline (Task 2)

Two-stage per-pathology SCRC with GNN-DRE weights:

1. **Stage 1** — shared across all strategies (β = 0.15 entropy deferral)
2. **Stage 2** — per-strategy α_k array fed to weighted CRC calibration


In [ ]:
# Stage 1 deferral is identical for all allocation strategies
entropy_cal = multilabel_entropy(p_cal)
defer_mask_cal = select_for_deferral(entropy_cal, BETA, seed=SEED)

entropy_nih = multilabel_entropy(p_nih)
defer_mask_nih = select_for_deferral(entropy_nih, BETA, seed=SEED)

kept_cal = ~defer_mask_cal
kept_nih = ~defer_mask_nih

print(f"Stage 1  (beta={BETA}):")
print(f"  Cal  deferred: {defer_mask_cal.sum():,} / {len(defer_mask_cal):,} "
      f"({defer_mask_cal.mean()*100:.1f}%)  kept: {kept_cal.sum():,}")
print(f"  NIH  deferred: {defer_mask_nih.sum():,} / {len(defer_mask_nih):,} "
      f"({defer_mask_nih.mean()*100:.1f}%)  kept: {kept_nih.sum():,}")


In [ ]:
print("Stage 2: per-pathology CRC calibration\n")
cal_results = {}

for method_name, alpha_arr in allocations.items():
    result = calibrate_per_pathology_crc_fnr(
        probs           = p_cal[kept_cal],
        labels          = Y_cal[kept_cal],
        weights         = w_cal[kept_cal],
        alpha           = alpha_arr,
        n_grid          = 1001,
        pathology_names = PATHOLOGIES,
    )
    cal_results[method_name] = result

    lam = result.lambda_hats
    fnr = result.weighted_fnr_at_lambda
    print(f"  [{method_name:<18}]  alpha_mean={alpha_arr.mean():.3f}"
          f"  lambda*=[{lam.min():.3f}, {lam.max():.3f}]"
          f"  cal_FNR=[{fnr.min():.3f}, {fnr.max():.3f}]")

print("\nCalibration complete.")


In [ ]:
def evaluate_fnr_fpr(probs_kept, labels_kept, lambda_stars):
    """Empirical (unweighted) FNR and FPR per pathology on retained test samples."""
    K   = probs_kept.shape[1]
    fnrs = np.full(K, np.nan)
    fprs = np.full(K, np.nan)
    for k in range(K):
        valid = ~np.isnan(labels_kept[:, k])
        yv    = labels_kept[valid, k]
        pv    = probs_kept [valid, k]
        pred  = (pv >= lambda_stars[k]).astype(float)
        pos, neg = (yv == 1), (yv == 0)
        if pos.sum() > 0:
            fnrs[k] = (pred[pos] == 0).mean()
        if neg.sum() > 0:
            fprs[k] = (pred[neg] == 1).mean()
    return fnrs, fprs


test_results = {}
print(f"{'Method':<20} {'Mean FNR':>10} {'Mean FPR':>10} {'Max FPR':>10}")
print("-" * 52)
for method_name, cal_result in cal_results.items():
    fnrs, fprs = evaluate_fnr_fpr(
        p_nih[kept_nih],
        Y_nih_test[kept_nih],
        cal_result.lambda_hats,
    )
    test_results[method_name] = {'fnrs': fnrs, 'fprs': fprs}
    print(f"  {method_name:<18} {np.nanmean(fnrs):>10.4f}"
          f" {np.nanmean(fprs):>10.4f} {np.nanmax(fprs):>10.4f}")


## 6. Results & Visualisation (Task 3)

### Metrics
- **Mean FPR** — system efficiency (↓ better)
- **Max FPR** — worst-case single-pathology failure (critical)
- **FNR Gap** — mean|Test_FNR_k − α_k| — calibration honesty
- **Clinical Cost** — mean(5·FNR_k + FPR_k) — clinical weighting (FNR costs 5×)


In [ ]:
def compute_metrics(name, alpha_arr, cal_res, test_res):
    fnrs, fprs = test_res['fnrs'], test_res['fprs']
    return {
        'Method'          : name,
        'Mean FPR ↓'      : round(float(np.nanmean(fprs)), 4),
        'Max FPR ↓'       : round(float(np.nanmax(fprs)),  4),
        'Mean FNR'        : round(float(np.nanmean(fnrs)), 4),
        'FNR Gap ↓'       : round(float(np.nanmean(np.abs(fnrs - alpha_arr))), 4),
        'Clinical Cost ↓' : round(float(np.nanmean(5*fnrs + fprs)), 4),
    }

rows = [compute_metrics(n, allocations[n], cal_results[n], test_results[n])
        for n in allocations]
metrics_df = pd.DataFrame(rows).set_index('Method')

print("## Evaluation Metrics\n")
try:
    print(metrics_df.to_markdown())
except ImportError:
    print(metrics_df.to_string())
print()
print("FNR Gap  = mean|Test_FNR_k - alpha_k|   (calibration honesty)")
print("Clin.Cost= mean(5*FNR_k + FPR_k)        (5x clinical weight on FNR)")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

N = K
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close polygon

short_labels = []
for p in PATHOLOGIES:
    lbl = p.replace('Pneumothorax', 'PneumoTx').replace('Pneumonia', 'Pneumonia')
    short_labels.append(lbl[:9])

method_styles = {
    'Uniform'        : ('#e74c3c', '-',  2.5, 0.12),
    'Linear'         : ('#f39c12', '--', 2.0, 0.08),
    'Inverse_Excess' : ('#2ecc71', '-',  2.5, 0.15),
    'Softmax'        : ('#3498db', ':',  2.0, 0.08),
}

for name, (color, ls, lw, fill_alpha) in method_styles.items():
    vals = test_results[name]['fprs'].tolist()
    vals = [0.0 if np.isnan(v) else v for v in vals]
    vals += vals[:1]
    ax.plot(angles, vals, color=color, linestyle=ls, linewidth=lw, label=name)
    ax.fill(angles, vals, color=color, alpha=fill_alpha)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(short_labels, size=12, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9, color='gray')
ax.set_title(
    'Test FPR per Pathology by Alpha Allocation Strategy\n'
    '(closer to centre = lower FPR = more efficient)',
    pad=30, size=14, fontweight='bold'
)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=11)
ax.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.4)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'scrc_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: report/scrc_radar_chart.png")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

styles = [
    ('Uniform',         '#e74c3c', 'o',  130),
    ('Inverse_Excess',  '#2ecc71', 's',  130),
    ('Linear',          '#f39c12', '^',  100),
    ('Softmax',         '#3498db', 'D',  100),
]
label_offsets = {
    'Uniform':        ( 7,   4),
    'Inverse_Excess': ( 7, -13),
    'Linear':         (-48,  6),
    'Softmax':        (-48,-13),
}

for name, color, marker, sz in styles:
    fnrs = test_results[name]['fnrs']
    fprs = test_results[name]['fprs']
    ax.scatter(fprs, fnrs, color=color, marker=marker, s=sz,
               label=name, zorder=5, edgecolors='k', linewidths=0.7)
    ox, oy = label_offsets[name]
    for k, (x, y) in enumerate(zip(fprs, fnrs)):
        if np.isnan(x) or np.isnan(y):
            continue
        ax.annotate(PATHOLOGIES[k][:5], (x, y),
                    textcoords='offset points', xytext=(ox, oy),
                    fontsize=8, color=color, fontweight='bold')

ax.axhline(ALPHA_TARGET, color='red', linestyle='--', linewidth=1.5,
           alpha=0.7, label=f'Target \u03b1 = {ALPHA_TARGET}')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.4)

ax.set_xlabel('Test FPR  (\u2193 better)', fontsize=13)
ax.set_ylabel('Test FNR  (\u2193 better, \u2264 \u03b1_k guaranteed at calibration)', fontsize=13)
ax.set_title('FNR vs FPR Trade-off by Alpha Allocation Strategy\n'
             '(ideal: bottom-left corner)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 0.65)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'scrc_fnr_fpr_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: report/scrc_fnr_fpr_scatter.png")


## 7. Beta Sweep Ablation (Task 4)

**Question**: How does the required deferral rate affect FPR on retained samples?

Sweep β ∈ {0.05, 0.10, …, 0.30} using **Inverse Excess** allocation.
Each β generates a different (Stage 1) deferral mask → different calibration and test subsets.
Expected: smooth Pareto curve — higher β → lower FPR on retained samples.


In [ ]:
betas = np.arange(0.05, 0.31, 0.05)
inv_excess_alphas = alloc_inverse_excess(gnn_aucs, ALPHA_TARGET)

print(f"Beta sweep: {np.round(betas, 2).tolist()}")
print(f"Inverse-Excess alpha array: {np.round(inv_excess_alphas, 3).tolist()}\n")

sweep_rows = []
for beta in betas:
    defer_cal_b = select_for_deferral(entropy_cal, beta, seed=SEED)
    defer_nih_b = select_for_deferral(entropy_nih, beta, seed=SEED)
    kept_cal_b  = ~defer_cal_b
    kept_nih_b  = ~defer_nih_b

    cal_b = calibrate_per_pathology_crc_fnr(
        probs           = p_cal[kept_cal_b],
        labels          = Y_cal[kept_cal_b],
        weights         = w_cal[kept_cal_b],
        alpha           = inv_excess_alphas,
        n_grid          = 1001,
        pathology_names = PATHOLOGIES,
    )
    fnrs_b, fprs_b = evaluate_fnr_fpr(
        p_nih[kept_nih_b],
        Y_nih_test[kept_nih_b],
        cal_b.lambda_hats,
    )
    row = {
        'beta'         : round(float(beta), 2),
        'n_deferred'   : int(defer_nih_b.sum()),
        'n_retained'   : int(kept_nih_b.sum()),
        'deferral_rate': round(float(defer_nih_b.mean()), 4),
        'mean_fnr'     : round(float(np.nanmean(fnrs_b)), 4),
        'mean_fpr'     : round(float(np.nanmean(fprs_b)), 4),
        'max_fpr'      : round(float(np.nanmax(fprs_b)),  4),
        'clinical_cost': round(float(np.nanmean(5*fnrs_b + fprs_b)), 4),
    }
    sweep_rows.append(row)
    print(f"  beta={beta:.2f}  retained={row['n_retained']:,}"
          f"  mean_fpr={row['mean_fpr']:.4f}  max_fpr={row['max_fpr']:.4f}")

sweep_df = pd.DataFrame(sweep_rows)
print("\n", sweep_df.to_string(index=False))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Pareto frontier (beta vs metric values)
ax1.plot(sweep_df['beta'], sweep_df['mean_fpr'],  'o-',
         color='#2ecc71', lw=2.5, ms=8, label='Mean FPR (retained)')
ax1.plot(sweep_df['beta'], sweep_df['max_fpr'],   's--',
         color='#e74c3c', lw=2.0, ms=8, label='Max FPR (retained)')
ax1.plot(sweep_df['beta'], sweep_df['mean_fnr'],  '^:',
         color='#3498db', lw=2.0, ms=8, label='Mean FNR (retained)')
ax1.axhline(ALPHA_TARGET, color='gray', linestyle=':', lw=1.5, alpha=0.6,
            label=f'\u03b1 = {ALPHA_TARGET}')
for _, row in sweep_df.iterrows():
    ax1.annotate(f"\u03b2={row['beta']:.2f}",
                 (row['beta'], row['mean_fpr']),
                 textcoords='offset points', xytext=(-5, 11), fontsize=8, color='#2ecc71')
ax1.set_xlabel('\u03b2 (Deferral Rate Budget)', fontsize=13)
ax1.set_ylabel('Metric on Retained Test Samples', fontsize=13)
ax1.set_title('Pareto Frontier: Deferral Rate vs FPR\n'
              '(Inverse Excess Allocation, \u03b1=0.10)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Right: samples retained vs mean FPR trade-off
ax2.plot(sweep_df['n_retained'], sweep_df['mean_fpr'], 'o-',
         color='#9b59b6', lw=2.5, ms=8)
for _, row in sweep_df.iterrows():
    ax2.annotate(f"\u03b2={row['beta']:.2f}",
                 (row['n_retained'], row['mean_fpr']),
                 textcoords='offset points', xytext=(6, 4), fontsize=9)
ax2.set_xlabel('Number of Retained Test Samples', fontsize=13)
ax2.set_ylabel('Mean Test FPR', fontsize=13)
ax2.set_title('Automation-Efficiency Trade-off\n'
              '(more retained \u2192 higher FPR, fewer deferred to expert)', fontsize=12)
ax2.grid(alpha=0.3)

plt.suptitle('Beta Sweep: Inverse Excess Alpha Allocation', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'scrc_beta_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: report/scrc_beta_sweep.png")


## Summary

### Key Findings

| Finding | Detail |
|---|---|
| **FPR collapse (Uniform)** | Uniform α=0.10 sets the same tight budget on near-random pathologies (Pneumothorax AUC≈0.63) as on confident ones (Effusion AUC≈0.83), forcing very low λ_k* → FPR spikes |
| **Inverse Excess fix** | α_k ∝ 1/(AUC_k−0.5) gives loose budgets to hard pathologies → higher λ_k* → controlled FPR |
| **Budget neutrality** | All strategies respect mean(α_k) = 0.10 at the system level |
| **Beta sweep** | Smooth Pareto curve confirms the system is highly controllable: ↑β → ↓FPR on retained |

### Clinical Interpretation

- **Stage 1** defers the β% most uncertain cases to a human radiologist
- **Stage 2** provides per-pathology FNR ≤ α_k guarantees on retained cases
- **Capability Alpha** prevents false alarm spikes on pathologies the model struggles with
- The Beta Pareto curve lets clinicians choose their preferred automation-accuracy trade-off

### Output Files

| File | Description |
|---|---|
| `report/scrc_radar_chart.png` | Radar plot of per-pathology FPR by allocation strategy |
| `report/scrc_fnr_fpr_scatter.png` | FNR vs FPR scatter (Uniform vs Inverse Excess) |
| `report/scrc_beta_sweep.png` | Pareto frontier: deferral rate vs FPR |
| `report/scrc_capability_alpha_report.md` | Markdown experiment report |
